# 🤖 AI-EMCS: Artificial Intelligence-Powered Environmental Monitoring and Control System Using Arduino

### 📌 Project Metadata
* **System Architecture:** Local Edge-AI Integration (Offline LLM Deployment)
* **Microcontroller Platform:** Arduino Embedded Architecture
* **Primary Features:** Real-Time Telemetry Logging, Intelligent Decision Making, Hardware Automation, and Remote Mobile Broadcasting via Telegram API

---

## 📖 Project Description
The **AI-EMCS** is an advanced engineering prototype designed to monitor indoor environment dynamics using a localized array of physical sensors. The system continuously tracks ambient **Temperature**, **Humidity**, **Carbon Dioxide ($CO_2$) gas concentrations**, and **Lux intensity levels**. 

Unlike traditional rule-based automated systems, AI-EMCS utilizes a **Local Large Language Model (TinyLlama via LM Studio)** acting as an on-site edge intelligence engine. The system processes raw telemetry data directly on the host computer without requiring external internet infrastructure, ensuring **zero cloud subscription costs**, **unlimited data query capacity**, and **absolute user data privacy**. 

Based on the parsed environmental conditions, the local AI dynamically categorizes the room safety status and transmits targeted feedback signals to the hardware layer (triggering automated tone frequencies for critical warnings or structural emergency hazards) while simultaneously broadcasting encrypted updates to the user's remote mobile terminal via the Telegram bot framework.

In [15]:
!pip install google-genai pyserial


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import serial
import json
import time
import requests
from openai import OpenAI

# === CONFIGURATION ===
ARDUINO_PORT = "COM10"
BAUD_RATE = 9600

# 🔑 TELEGRAM CONFIGURATION (For Version 2)
TELEGRAM_BOT_TOKEN = "YOUR_TELEGRAM_BOT_TOKEN"
TELEGRAM_CHAT_ID = "YOUR_TELEGRAM_CHAT_ID"

try:
    arduino = serial.Serial(ARDUINO_PORT, BAUD_RATE, timeout=1)
    time.sleep(2)
    print(f"=== AI-EMCS CONTROL CENTER: CONNECTED TO {ARDUINO_PORT} ===")
    
    client = OpenAI(
        base_url="http://127.0.0.1:1234/v1",
        api_key="lm-studio"
    )
    print("=== LM STUDIO LOCAL LLM: INITIALIZED & READY (OFFLINE MODE) ===\n")
except Exception as e:
    print(f"Error sa Initialization: {e}")
    exit()

def send_telegram_alert(message):
    url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage"
    payload = {"chat_id": TELEGRAM_CHAT_ID, "text": message}
    try:
        requests.post(url, json=payload, timeout=5)
    except Exception:
        pass

def hingi_ng_rekomendasyon_sa_local_ai(status_type, description):
    
    # 🛠️ STRICT MATHEMATICAL TEMPLATE INJECTION (Python handles the accuracy)
    if status_type == "HAZARD":
        exact_advice = "High CO2 detected! Open windows immediately."
    elif "dark and hot" in description.lower():
        exact_advice = "Dark and hot room! Please turn on lights and fan."
    elif "temperature" in description.lower():
        exact_advice = "High temperature! Please turn on the fan or AC."
    elif "visibility" in description.lower() or "light" in description.lower():
        exact_advice = "Low visibility! Please switch on the indoor lights."
    else:
        exact_advice = "Room conditions are normal and comfortable."
        
    system_instruction = (
        "You are an AI system. Your only task is to rewrite the given text shortly. "
        "Do not chat, do not explain, do not add introductory phrases like 'Safety Text:'."
    )
    
    user_prompt = f"Text to output: {exact_advice}"
    
    try:
        response = client.chat.completions.create(
            model="tinyllama-1.1b-chat-v1.0", 
            messages=[
                {"role": "system", "content": system_instruction},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.0, # Lock to zero variability
            max_tokens=40
        )
        ai_phrase = response.choices[0].message.content.strip()
        
        # 🧹 UPGRADED HARD CLEANING LAYER
        clean_tags = [
            "Sure, here's an AI-generated response to your text:",
            "Sure, here's a short safety command sentence based on the issue:",
            "Sure, here's an example safety command sentence based on the issue:",
            "Safety Text:", "Text to output:", "Command Advice:", "Command:", 
            "Safety Command:", "Here's the output:", "! the output for your provided safety text:"
        ]
        
        for tag in clean_tags:
            if tag.lower() in ai_phrase.lower():
                import re
                insensitive_tag = re.compile(re.escape(tag), re.IGNORECASE)
                ai_phrase = insensitive_tag.sub("", ai_phrase).strip()

        ai_phrase = ai_phrase.replace('\"', '').replace('\n', ' ').strip()
        
        if len(ai_phrase) < 5 or "stable and secure" in ai_phrase.lower() and status_type == "WARNING":
            ai_phrase = exact_advice

        return f"[{status_type}] {ai_phrase}"
        
    except Exception:
        return f"[{status_type}] {exact_advice}"
last_status = ""

# === MAIN LOOP ===
while True:
    try:
        if arduino.in_waiting > 0:
            raw_data = arduino.readline().decode('utf-8').strip()
            
            try:
                data = json.loads(raw_data)
                temp = data.get("temp", 0.0)
                hum = data.get("hum", 0.0)
                co2 = data.get("co2", 0)
                light = data.get("light", 0)
                
                print(f"\n[SENSOR DATA] Temp: {temp}°C | Hum: {hum}% | CO2: {co2} | Light: {light}%")
                
                # 🛠️ HARDWARE MATHEMATICAL LOGIC 
                status_type = "SAFE"
                description = "Conditions normal."
                
                if co2 > 200:
                    status_type = "HAZARD"
                    description = "High gas levels."
                elif temp > 34.0 and light < 20:
                    status_type = "WARNING"
                    description = "Dark and hot room."
                elif temp > 34.0:
                    status_type = "WARNING"
                    description = "High room temperature."
                elif light < 20:
                    status_type = "WARNING"
                    description = "Low light visibility."

                print("Sending to LM Studio... Local AI is thinking...")
                recommendation = hingi_ng_rekomendasyon_sa_local_ai(status_type, description)
                print(f"[LOCAL AI RECOMMENDATION]:\n{recommendation}")
                
                if status_type == "HAZARD":
                    print("-> AI Status: HAZARD! Deploying rapid chime loops.")
                    arduino.write(b'H')
                elif status_type == "WARNING":
                    print("-> AI Status: WARNING! Deploying standard alert pulse.")
                    arduino.write(b'W')
                else:
                    print("-> AI Status: SAFE. Silencing audio actuators.")
                    arduino.write(b'S')
                
                # Telegram sync (For version 2, only send if there's a status change and not in testing mode)
                if status_type != last_status and "YOUR_TELEGRAM" not in TELEGRAM_BOT_TOKEN:
                    telegram_text = (
                        f"📢 AI-EMCS UPDATE:\n"
                        f"📊 Readings: {temp}°C | {hum}% Hum | CO2: {co2} | Light: {light}%\n"
                        f"🤖 AI Advice: {recommendation}"
                    )
                    send_telegram_alert(telegram_text)
                    last_status = status_type
                    
                print("-" * 65)
                time.sleep(15) # Delay between readings to prevent spamming and allow for sensor stabilization
                
            except json.JSONDecodeError:
                continue
                
    except KeyboardInterrupt:
        print("\n[INFO] AI Control Center has been shut down successfully.")
        arduino.close()
        break

=== AI-EMCS CONTROL CENTER: CONNECTED TO COM10 ===
=== LM STUDIO LOCAL LLM: INITIALIZED & READY (OFFLINE MODE) ===


[SENSOR DATA] Temp: 35.2°C | Hum: 55.0% | CO2: 48 | Light: 85%
Sending to LM Studio... Local AI is thinking...
[LOCAL AI RECOMMENDATION]:
[WARNING] High temperature! Please turn on the fan or AC.
-> AI Status: WARNING! Deploying standard alert pulse.
-----------------------------------------------------------------

[SENSOR DATA] Temp: 35.2°C | Hum: 54.0% | CO2: 37 | Light: 0%
Sending to LM Studio... Local AI is thinking...
[LOCAL AI RECOMMENDATION]:
[WARNING] Dark and hottest room! Please turn on lights and fan.
-> AI Status: WARNING! Deploying standard alert pulse.
-----------------------------------------------------------------

[SENSOR DATA] Temp: 35.6°C | Hum: 63.0% | CO2: 34 | Light: 86%
Sending to LM Studio... Local AI is thinking...
[LOCAL AI RECOMMENDATION]:
[WARNING] High temperature! Please turn on the fan or AC.
-> AI Status: WARNING! Deploying standard aler